In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

In [50]:
class OvercookedKitchen(gym.Env):
    def __init__(self, layout):
        self.layout = layout
        self.height = len(layout)
        self.width = len(layout[0])
        self.TIME_TO_CHOP = 3
        self.TIME_TO_COOK = 5
        self.MAX_STEPS = 200

        # Actions : 0-3 Movement, 4 Interact, 5 Chop
        self.action_space = spaces.Discrete(6)
        # Observation space : agent_pos (2), agent_dir (2), held_obj (1), pans_state (number of pans),
        # dishes_state (number of dishes * 3), chopping_boards_state (number of chopping boards), Timer (1)
        # low=-1 because we can have -1 for the held object (when we hold nothing)
        self.observation_space = spaces.Box(low=-1, high=max(self.width, self.height, self.MAX_STEPS), shape=(11,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # We reset the timer
        self.step_count = 0
        # We place the agent in the middle of the kitchen, facing downwards
        self.agent_pos = (2, 2)
        self.agent_dir = (0, 1) # Regarde vers le bas
        self.held_obj = None
        self.done = False
        
        self.objects_on_counters = {} # {(x, y): obj_dict}
        self.pans_state = {}          # {(x, y): {'status': 'empty', 'timer': 0}}
        self.dishes = {}              # {(x, y): {'bread': False, 'tomato': False, 'steak': False}}
        self.chopping_boards = {}     # {(x, y): progression}
        
        # Milestone tracking for reward shaping: each sub-task is rewarded only once per episode
        self.milestones = set()
        
        for y, row in enumerate(self.layout):
            for x, tile in enumerate(row):
                if tile == 'P':
                    self.pans_state[(x, y)] = {'status': 'empty', 'timer': 0}
                elif tile == 'D':
                    self.dishes[(x, y)] = {'bread': False, 'tomato': False, 'steak': False}
                elif tile == 'C':
                    self.chopping_boards[(x, y)] = 0 
        return self._get_obs(), {}

    def _milestone_reward(self, milestone_name, reward_value):
        """Return reward_value the first time a milestone is reached, 0 afterwards."""
        if milestone_name not in self.milestones:
            self.milestones.add(milestone_name)
            return reward_value
        return 0.0

    def render(self):
        grid_copy = [list(row) for row in self.layout]
        
        # We display the objects on the counters (tomatoes, steaks, bread) with their first letter in uppercase
        for (x, y), obj in self.objects_on_counters.items():
            char = obj[0].upper() if isinstance(obj, str) else obj['name'][0].upper()
            grid_copy[y][x] = char

        # We display the agent with an arrow indicating its direction
        ax, ay = self.agent_pos
        dir_map = {(0, -1): '^', (0, 1): 'v', (-1, 0): '<', (1, 0): '>'}
        grid_copy[ay][ax] = dir_map.get(self.agent_dir, 'A')

        # Clean display
        print("\n" + "="*20)
        for row in grid_copy:
            print(" ".join(row))
        print(f"Hand : {self.held_obj}")
        print(f"Pans : {[(p['status'], p['timer']) for p in self.pans_state.values()]}")
        print(f"Dishes : {[(d['bread'], d['tomato'], d['steak']) for d in self.dishes.values()]}")
        print(f"Chopping boards : {[(c, self.chopping_boards[c]) for c in self.chopping_boards]}")
        print(f"Objects on counters : {self.objects_on_counters}")
        print(f"Steps: {self.step_count}/{self.MAX_STEPS}")
        print("="*20)
    
    def _get_obs(self):
        """ 
        Get the current observation of the environment. We will encode:
        1. Agent's position and direction
        2. Held object (if any)
        3. State of the pans (pan_code: 0=empty, 1-4=cooking timer, 5=ready)
        4. State of the dishes (which ingredients are on them)
        5. State of chopping boards (0=empty, 1-3=progression)
        6. Current step count
        """
        # Agents's position and direction
        agent_info = [self.agent_pos[0], self.agent_pos[1], self.agent_dir[0], self.agent_dir[1]]
        
        # Held object encoding (0: None, 1: Tomato, 2: Steak, 3: Bread, 4: Chopped Tomato, 5: Cooked Steak, 6: Full Burger)
        obj_map = {None: 0, "tomato": 1, "steak": 2, "bread": 3, 
                "chopped_tomato": 4, "cooked_steak": 5, "full_burger": 6}
        
        # If held_obj is a dict (i.e. an object on a counter), we get its name to encode it, otherwise we encode directly the held_obj string
        if isinstance(self.held_obj, dict):
            held_val = obj_map.get(self.held_obj.get('name'), 0)
        else:
            held_val = obj_map.get(self.held_obj, 0)
        
        # State of the pans: encode as 0 (empty), 1-4 (cooking timer), 5 (ready)
        # This removes observation aliasing: timer=5 always means ready
        pan_states = []
        for pos in self.pans_state:
            pan = self.pans_state[pos]
            if pan['status'] == 'ready':
                pan_states.append(self.TIME_TO_COOK)  # Distinct "ready" code
            else:
                pan_states.append(pan['timer'])  # 0 = empty, 1..TIME_TO_COOK-1 = cooking
            
        dish_states = []
        for pos in self.dishes:
            d = self.dishes[pos]
            # 1 if the ingredient is present, 0 otherwise
            dish_states.extend([int(d['bread']), int(d['tomato']), int(d['steak'])])

        # Chopping board state reflects actual progression (fixed sync from _handle_chopping)
        chop_states = [self.chopping_boards[pos] for pos in self.chopping_boards]

        # We add the timer
        time_left = [float(self.step_count)]

        # We concatenate all the information into a single observation vector for Gymnasium
        obs = np.array(agent_info + [held_val] + pan_states + dish_states + chop_states + time_left, dtype=np.float32)
        return obs
    
    def _get_adj_pos(self):
        """Calculate the position adjacent to the agent based on its current direction."""
        x, y = self.agent_pos
        dx, dy = self.agent_dir
        return (x + dx, y + dy)

    def step(self, action):
        reward = -0.01 # Survival penalty to encourage efficiency

        ## Cooking progress update (passive cooking)
        for _, pan in self.pans_state.items():
            if pan['status'] == 'cooking':
                pan['timer'] += 1
                if pan['timer'] >= self.TIME_TO_COOK:
                    pan['status'] = 'ready'
                    # Reward the agent when its steak finishes cooking
                    reward += self._milestone_reward('cooked_steak_ready', 1.0)

        if action < 4: # MOVEMENT
            self._handle_move(action)
        
        elif action == 4: # INTERACT (Take/Place/Cook)
            target_pos = self._get_adj_pos()
            tile = self.layout[target_pos[1]][target_pos[0]]
            
            if tile == 'T': # Tomatoes
                if self.held_obj is None:
                    self.held_obj = "tomato"
                    reward += self._milestone_reward('picked_tomato', 0.5)
            elif tile == 'S': # Steak
                if self.held_obj is None:
                    self.held_obj = "steak"
                    reward += self._milestone_reward('picked_steak', 0.5)
            elif tile == 'B': # Bread
                if self.held_obj is None:
                    self.held_obj = "bread"
                    reward += self._milestone_reward('picked_bread', 0.5)
            elif tile == 'P': # Pan
                reward += self._handle_pan_interact(target_pos)
            elif tile == ' ': # Empty space (BUG FIX: was '' which never matched)
                self._handle_empty_space_interact(target_pos)
            elif tile == 'D': # Dish
                reward += self._handle_dish_interact(target_pos)
            elif tile == 'C': # Chopping board
                reward += self._handle_chopping_board_interact(target_pos)
            elif tile == 'V': # Delivery point
                delivery_reward = self._handle_delivery()
                reward += delivery_reward 
                
        elif action == 5: # Chop
            target_pos = self._get_adj_pos()
            if self.layout[target_pos[1]][target_pos[0]] == 'C': # Chopping board
                reward += self._handle_chopping(target_pos)

        self.step_count += 1
        if self.step_count >= self.MAX_STEPS:
            self.done = True

        return self._get_obs(), reward, self.done, False, {}
    
    def _handle_move(self, action):
        # We map the action to a direction vector
        # 0: Up, 1: Down, 2: Left, 3: Right
        directions = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        dx, dy = directions[action]
        
        # We update the agent's direction
        self.agent_dir = (dx, dy)
        
        new_x = self.agent_pos[0] + dx
        new_y = self.agent_pos[1] + dy
        
        if 0 <= new_x < self.width and 0 <= new_y < self.height:
            # We can only move if the target tile is empty
            if self.layout[new_y][new_x] == ' ':
                self.agent_pos = (new_x, new_y)

    def _handle_chopping_board_interact(self, target_pos):
        obj_on_board = self.objects_on_counters.get(target_pos)
        reward = 0.0

        if obj_on_board is None and self.held_obj == "tomato":
            # We put the tomato on the chopping board to start chopping
            self.objects_on_counters[target_pos] = {
                'name': 'tomato',
                'status': 'raw',
                'progression': 0
            }
            self.held_obj = None
            self.chopping_boards[target_pos] = 0
            reward += self._milestone_reward('placed_tomato_on_board', 0.5)

        elif obj_on_board is not None and obj_on_board['status'] == 'chopped' and self.held_obj is None:
            self.held_obj = "chopped_tomato"
            del self.objects_on_counters[target_pos]
            self.chopping_boards[target_pos] = 0 # Reset the chopping board display
            reward += self._milestone_reward('picked_chopped_tomato', 0.5)
            
        return reward

    def _handle_empty_space_interact(self, target_pos):
        # We check if there's an object on the target tile
        obj_on_tile = self.objects_on_counters.get(target_pos)

        if self.held_obj is not None:
            # The agent holds an object
            if obj_on_tile is None:
                # Free tile : we can place the object
                self.objects_on_counters[target_pos] = self.held_obj
                self.held_obj = None

        elif obj_on_tile is not None:
            # The agent has empty hands and there is an object on the tile : we pick it up
            self.held_obj = obj_on_tile
            del self.objects_on_counters[target_pos]

    def _handle_pan_interact(self, target_pos):
        pan = self.pans_state.get(target_pos) # Ex: {'status': 'empty', 'timer': 0}
        reward = 0.0

        if self.held_obj == "steak" and pan['status'] == 'empty':
            # We place the steak in the pan to cook it
            pan['status'] = 'cooking'
            pan['timer'] = 0
            self.held_obj = None
            reward += self._milestone_reward('placed_steak_in_pan', 0.5)
            
        elif pan['status'] == 'ready' and self.held_obj is None:
            # We pick up the cooked steak
            self.held_obj = "cooked_steak"
            pan['status'] = 'empty'
            pan['timer'] = 0
            reward += self._milestone_reward('picked_cooked_steak', 0.5)

        return reward
    
    def _handle_dish_interact(self, target_pos):
        dish = self.dishes.get(target_pos) # Ex: {'bread': False, 'tomato': False, 'steak': False}
        reward = 0.0
        
        if self.held_obj:
            if self.held_obj == "bread" and not dish['bread']:
                dish['bread'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_bread_on_dish', 2.0)
            elif self.held_obj == "chopped_tomato" and not dish['tomato'] and dish['bread']:
                dish['tomato'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_tomato_on_dish', 2.0)
            elif self.held_obj == "cooked_steak" and not dish['steak'] and dish['bread']:
                dish['steak'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_steak_on_dish', 2.0)

        # If the dish is complete, we can pick up the full burger
        elif all(dish.values()):
            self.held_obj = "full_burger"
            # Reset the dish for the next order
            for key in dish: dish[key] = False
            reward += self._milestone_reward('assembled_burger', 2.0)

        return reward

    def _handle_chopping(self, target_pos):
        # We check if an object is on the chopping board at the target position
        obj = self.objects_on_counters.get(target_pos)
        reward = 0.0
        
        if obj and obj['name'] == 'tomato' and obj['status'] == 'raw':
            obj['progression'] += 1
            # Sync the chopping board observation with actual progression (BUG FIX)
            self.chopping_boards[target_pos] = obj['progression']
            if obj['progression'] >= self.TIME_TO_CHOP:
                obj['status'] = 'chopped'
                reward += self._milestone_reward('chopped_tomato_done', 1.0)

        return reward

    def _handle_delivery(self):
        if self.held_obj == "full_burger":
            reward = 10.0 # Reward for delivering a complete burger
            self.held_obj = None
            self.done = True # We end the episode at this point
        else:
            # Increased penalty for trying to deliver incomplete or wrong items
            reward = -1.0
        return reward


In [51]:
# Layout Legend:
# X: Wall/Counter (impassable)
# T: Tomatoes, S: Steaks, B: Bread
# C: Chopping board, P: Pan, D: Dish
#  : Empty space (where one can walk and place objects)

MY_LAYOUT = [
    "X T S B X",
    "C       P",
    "X       X",
    "X   D   V",
    "XXXXXXXXX"
]

In [52]:
env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()

for _ in range(1000):
    action = env.action_space.sample() # Random Action
    next_obs, reward, done, truncated, info = env.step(action)
    
    # Integrity check
    assert env.observation_space.contains(next_obs), "Out of bounds observation!"
    
    if done:
        obs, _ = env.reset()

## Tests

In [ ]:
## CHOP TOMATOES AND PLACE THEM ON DISH TEST

env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
env.render()

# Simulate a specific sequence of actions:
# Reminder: 0: Up, 1: Down, 2: Left, 3: Right, 4: Interact, 5: Chop
# Here, we will go to the tomato, take it, go to the chopping board, chop it, then place it on the dish, and come back to the start
# Note: as there is no bread, this sequence doesn't place the tomatoe
actions_tomatoes_on_dish = [0, 0, 4, 2, 2, 4, 5, 5, 5, 4, 1, 1, 3, 3, 4, 0, 2] 

for a in actions_tomatoes_on_dish:
    obs, reward, done, _, _ = env.step(a)
    env.render()
    print(f"Reward received : {reward}")
    if done: break


X   T   S   B   X
C               P
X   v           X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 0/200

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 1/200
Reward received : -0.01

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 2/200
Reward received : -0.01

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : tomato
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 3/200
Reward received : -0.01

X   T   S   B

In [64]:
## COOK STEAK TEST 

env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
env.render()

# Simulate a specific sequence of actions:
# Reminder: 0: Up, 1: Down, 2: Left, 3: Right, 4: Interact, 5: Chop
# Here, we will go to the steak, take it, go to the pan, cook it, then place it on the dish, and come back to the start
# Note: as there is no bread, this sequence doesn't place the steak
actions_steak_on_dish = [0, 3, 3, 0, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 1, 1, 2, 2, 4, 0, 2, 2, 2] 

for a in actions_steak_on_dish:
    obs, reward, done, _, _ = env.step(a)
    env.render()
    print(f"Reward received : {reward}")
    if done: break


X   T   S   B   X
C               P
X   v           X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 0/200

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 1/200
Reward received : -0.01

X   T   S   B   X
C     >         P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 2/200
Reward received : -0.01

X   T   S   B   X
C       >       P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 3/200
Reward received : -0.01

X   T   S   B  

In [68]:
## PUT BREAD ON DISH TEST

env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
env.render()

# Simulate a specific sequence of actions:
# Reminder: 0: Up, 1: Down, 2: Left, 3: Right, 4: Interact, 5: Chop
actions_bread_on_dish = [0, 3, 3, 3, 3, 0, 4, 1, 1, 1, 2, 2, 4, 0, 2, 2, 2] 

for a in actions_bread_on_dish:
    obs, reward, done, _, _ = env.step(a)
    env.render()
    print(f"Reward received : {reward}")
    if done: break


X   T   S   B   X
C               P
X   v           X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 0/200

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 1/200
Reward received : -0.01

X   T   S   B   X
C     >         P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 2/200
Reward received : -0.01

X   T   S   B   X
C       >       P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 3/200
Reward received : -0.01

X   T   S   B  

In [ ]:
## FINAL TEST: COMPLETE A BURGER AND DELIVER IT

env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
env.render()

# Simulate a specific sequence of actions:
# Reminder: 0: Up, 1: Down, 2: Left, 3: Right, 4: Interact, 5: Chop
# Delivery step: get the burger, go to the delivery point and interact
actions_delivery = [3, 3, 1, 4, 3, 3, 3, 1, 3, 4] 
actions_test = actions_bread_on_dish + actions_tomatoes_on_dish + actions_steak_on_dish + actions_delivery
for a in actions_test:
    obs, reward, done, _, _ = env.step(a)
    env.render()
    print(f"Reward received : {reward}")
    if done: break


X   T   S   B   X
C               P
X   v           X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 0/200

X   T   S   B   X
C   ^           P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 1/200
Reward received : -0.01

X   T   S   B   X
C     >         P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 2/200
Reward received : -0.01

X   T   S   B   X
C       >       P
X               X
X       D       V
X X X X X X X X X
Hand : None
Pans : [('empty', 0)]
Dishes : [(False, False, False)]
Chopping boards : [((0, 1), 0)]
Objects on counters : {}
Steps: 3/200
Reward received : -0.01

X   T   S   B  